In [ ]:
# ============================================================
# NOTEBOOK 1: DATASET PREPARATION
# Run this once — saves all datasets to /kaggle/working/datasets/
# ============================================================

import os
import json
import random
import pandas as pd
import numpy as np
import requests
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Directory setup
base_path = '/kaggle/working/datasets'
os.makedirs(base_path, exist_ok=True)
os.makedirs(f'{base_path}/alpaca', exist_ok=True)
os.makedirs(f'{base_path}/oa', exist_ok=True)

print("="*60)
print("DATASET PREPARATION")
print("="*60)

# ============================================================
# 1. PILE DATASET (Streaming, 50k samples)
# ============================================================
print("\n[1/5] Loading PILE dataset (streaming)...")

pile_save_path = f'{base_path}/pile_50k.csv'

if os.path.exists(pile_save_path):
    print(f"   ✅ Already exists: {pile_save_path}")
    pile_df = pd.read_csv(pile_save_path)
    print(f"   Samples: {len(pile_df)}")
else:
    parts = [
        {'skip': 0,      'target': 17000, 'label': 'Start'},
        {'skip': 200000, 'target': 17000, 'label': 'Middle'},
        {'skip': 500000, 'target': 16000, 'label': 'End'},
    ]

    all_samples = []

    for part in parts:
        print(f"\n   📥 Collecting from {part['label']} (skip={part['skip']})...")

        dataset = load_dataset(
            "monology/pile-uncopyrighted",
            split="train",
            streaming=True
        )

        samples = []
        skipped = 0
        processed = 0

        for example in dataset:
            if skipped < part['skip']:
                skipped += 1
                continue

            text = example.get('text', '').strip()
            if len(text) > 100:
                samples.append({
                    'text': text,
                    'source': example.get('meta', {}).get('pile_set_name', 'unknown')
                })
                processed += 1

            if processed >= part['target']:
                break

            if processed % 5000 == 0 and processed > 0:
                print(f"      Collected {processed}/{part['target']}...")

        print(f"   ✅ Collected {len(samples)} from {part['label']}")
        all_samples.extend(samples)

    print(f"\n   🔀 Shuffling {len(all_samples)} samples...")
    random.shuffle(all_samples)

    pile_df = pd.DataFrame(all_samples[:50000])
    pile_df.to_csv(pile_save_path, index=False)
    print(f"   ✅ PILE saved: {len(pile_df)} samples → {pile_save_path}")
    print(f"   Source distribution:\n{pile_df['source'].value_counts().head(5)}")

# ============================================================
# 2. T-REx DATASET (KILT, streaming download)
# ============================================================
print("\n[2/5] Loading T-REx (KILT) dataset...")

trex_save_path = f'{base_path}/trex_50k.csv'

if os.path.exists(trex_save_path):
    print(f"   ✅ Already exists: {trex_save_path}")
    trex_df = pd.read_csv(trex_save_path)
    print(f"   Samples: {len(trex_df)}")
else:
    url = "http://dl.fbaipublicfiles.com/KILT/trex-train-kilt.jsonl"
    print(f"   Downloading from: {url}")
    response = requests.get(url, stream=True)

    samples = []
    target = 50000
    raw_lines = 0

    for line in response.iter_lines():
        if len(samples) >= target:
            break
        if not line:
            continue

        raw_lines += 1
        try:
            item = json.loads(line)
            input_text = item.get('input', '').strip()
            outputs = item.get('output', [])
            if not outputs:
                continue
            answer = outputs[0].get('answer', '').strip()
            if not answer:
                continue
            if '[SEP]' not in input_text:
                continue

            parts = input_text.split('[SEP]')
            subject = parts[0].strip()
            relation = parts[1].strip() if len(parts) > 1 else ''
            text = f"{subject}'s {relation} is"
            entity = answer

            samples.append({
                'text': text,
                'entity': entity,
                'subject': subject,
                'relation': relation
            })

        except Exception:
            continue

        if raw_lines % 100000 == 0:
            print(f"   Processed {raw_lines} lines, collected {len(samples)}...")

    response.close()
    trex_df = pd.DataFrame(samples[:target])
    trex_df.to_csv(trex_save_path, index=False)
    print(f"   ✅ T-REx saved: {len(trex_df)} samples → {trex_save_path}")

# ============================================================
# 3. MMLU DATASET
# ============================================================
print("\n[3/5] Loading MMLU dataset...")

mmlu_test_path = f'{base_path}/mmlu_test.csv'
mmlu_val_path  = f'{base_path}/mmlu_val.csv'

if os.path.exists(mmlu_test_path) and os.path.exists(mmlu_val_path):
    print(f"   ✅ Already exists")
    mmlu_df     = pd.read_csv(mmlu_test_path)
    mmlu_val_df = pd.read_csv(mmlu_val_path)
    print(f"   Test samples:  {len(mmlu_df)}")
    print(f"   Val samples:   {len(mmlu_val_df)}")
else:
    mmlu     = load_dataset("cais/mmlu", "all", split="test")
    mmlu_val = load_dataset("cais/mmlu", "all", split="validation")

    mmlu_df     = pd.DataFrame(mmlu)
    mmlu_val_df = pd.DataFrame(mmlu_val)

    mmlu_df.to_csv(mmlu_test_path, index=False)
    mmlu_val_df.to_csv(mmlu_val_path, index=False)
    print(f"   ✅ MMLU test saved:  {len(mmlu_df)} samples")
    print(f"   ✅ MMLU val saved:   {len(mmlu_val_df)} samples")
    print(f"   Subjects: {mmlu_df['subject'].nunique()}")

# ============================================================
# 4. ALPACA DATASET (Fine-tuning data)
# ============================================================
print("\n[4/5] Loading Alpaca dataset...")

alpaca_path = f'{base_path}/alpaca/alpaca_data.csv'

if os.path.exists(alpaca_path):
    print(f"   ✅ Already exists")
    alpaca_df = pd.read_csv(alpaca_path)
    print(f"   Samples: {len(alpaca_df)}")
else:
    alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train")

    def format_alpaca(example):
        if example["input"] and example["input"].strip():
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Input:\n{example['input']}\n"
                f"### Response:\n{example['output']}"
            )
        else:
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Response:\n{example['output']}"
            )
        return {"text": text}

    alpaca_formatted = alpaca_raw.map(format_alpaca)
    alpaca_df = pd.DataFrame({
        'text': alpaca_formatted['text'],
        'instruction': alpaca_formatted['instruction'],
        'input': alpaca_formatted['input'],
        'output': alpaca_formatted['output'],
    })
    alpaca_df.to_csv(alpaca_path, index=False)
    print(f"   ✅ Alpaca saved: {len(alpaca_df)} samples → {alpaca_path}")

# ============================================================
# 5. OPENASSISTANT (OA) DATASET (Fine-tuning data)
# ============================================================
print("\n[5/5] Loading OpenAssistant (OA) dataset...")

oa_path = f'{base_path}/oa/oa_data.csv'

if os.path.exists(oa_path):
    print(f"   ✅ Already exists")
    oa_df = pd.read_csv(oa_path)
    print(f"   Samples: {len(oa_df)}")
else:
    oa_raw = load_dataset("OpenAssistant/oasst1", split="train")

    # Paper: extract single-turn English conversations
    oa_samples = []
    for example in oa_raw:
        if (example.get('lang') == 'en' and
            example.get('role') == 'prompter' and
            example.get('text')):

            text = (
                f"### Instruction:\n{example['text']}\n"
                f"### Response:\n"
            )
            oa_samples.append({
                'text': text,
                'raw_text': example['text'],
                'message_id': example.get('message_id', ''),
            })

    oa_df = pd.DataFrame(oa_samples)
    # Paper uses 11k pairs — sample down
    oa_df = oa_df.sample(n=min(11000, len(oa_df)), random_state=SEED).reset_index(drop=True)
    oa_df.to_csv(oa_path, index=False)
    print(f"   ✅ OA saved: {len(oa_df)} samples → {oa_path}")

# For fair comparison, sample Alpaca to same size as OA (paper does this)
alpaca_sampled_path = f'{base_path}/alpaca/alpaca_sampled.csv'
if not os.path.exists(alpaca_sampled_path):
    alpaca_sampled = alpaca_df.sample(
        n=min(11000, len(alpaca_df)),
        random_state=SEED
    ).reset_index(drop=True)
    alpaca_sampled.to_csv(alpaca_sampled_path, index=False)
    print(f"   ✅ Alpaca sampled to {len(alpaca_sampled)} (matching OA size)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("ALL DATASETS READY")
print("="*60)
print(f"  PILE       : {pile_save_path}")
print(f"  T-REx      : {trex_save_path}")
print(f"  MMLU test  : {mmlu_test_path}")
print(f"  MMLU val   : {mmlu_val_path}")
print(f"  Alpaca     : {alpaca_path}")
print(f"  Alpaca(11k): {alpaca_sampled_path}")
print(f"  OA         : {oa_path}")

In [ ]:
# ============================================================
# NOTEBOOK 2 PART 1: Setup + CLM + Facts Generation
# ============================================================

import os, gc, json, ast, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Verify GPUs ──────────────────────────────────────────────
print("GPU Check:")
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

# ── Paths & Config ───────────────────────────────────────────
MODEL_PATH   = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1'
DATA_PATH    = "/kaggle/working/datasets"
RESULTS_PATH = "/kaggle/working/results"
os.makedirs(RESULTS_PATH, exist_ok=True)

SEED         = 42
EVAL_SAMPLES = 30000
N_BINS       = 10

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

# ── Load Model ───────────────────────────────────────────────
print("\nLoading LLaMA-7B (fp16) across both T4 GPUs...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f"\nModel loaded!")
print(f"Device map (first 5 layers):")
for k, v in list(model.hf_device_map.items())[:5]:
    print(f"  {k}: {v}")
print(f"\nGPU Memory after loading:")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {used:.2f} / {total:.2f} GB used")

# ── ECE Utilities ────────────────────────────────────────────
def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

# ── Load Evaluation Datasets ─────────────────────────────────
print("\nLoading evaluation datasets...")
pile_df = pd.read_csv(f'{DATA_PATH}/pile_50k.csv')
trex_df = pd.read_csv(f'{DATA_PATH}/trex_50k.csv')

pile_eval = pile_df.sample(
    n=EVAL_SAMPLES, random_state=SEED
).reset_index(drop=True)

trex_eval = trex_df.sample(
    n=EVAL_SAMPLES, random_state=SEED
).reset_index(drop=True)

print(f"  PILE eval samples : {len(pile_eval)}")
print(f"  T-REx eval samples: {len(trex_eval)}")

# ── TASK 1: CLM (PILE) ───────────────────────────────────────
print("\n" + "="*60)
print("TASK 1: Causal Language Modeling (PILE)")
print("="*60)

# Baseline: no instruction prompt
clm_confidences = []
clm_accuracies  = []

clm_start_time = time.time() # Added timer start
for idx, row in pile_eval.iterrows():
    text = str(row['text']).strip()
    try:
        inputs    = tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=512
        )
        input_ids = inputs["input_ids"]
        seq_len   = input_ids.shape[1]
        if seq_len < 3:
            continue

        pos        = np.random.randint(1, seq_len)
        context    = input_ids[:, :pos].to(model.device)
        true_token = input_ids[:, pos].item()

        with torch.no_grad():
            out   = model(input_ids=context)
            probs = F.softmax(
                out.logits[:, -1, :].float(), dim=-1
            )

        clm_confidences.append(probs.max(dim=-1).values.item())
        clm_accuracies.append(
            int(probs.argmax(dim=-1).item() == true_token)
        )

    except Exception as e:
        continue

    # Changed 5000 to 1000 for faster updates, added percentage and timer
    if (idx + 1) % 1000 == 0:
        elapsed_mins = (time.time() - clm_start_time) / 60
        pct = (idx + 1) / EVAL_SAMPLES * 100
        print(f"  [{idx+1}/{EVAL_SAMPLES}] ({pct:.1f}%) | Time: {elapsed_mins:.1f}m | "
              f"ECE: {compute_ece(clm_confidences, clm_accuracies):.4f} | "
              f"Acc: {np.mean(clm_accuracies):.4f}")

clm_ece = compute_ece(clm_confidences, clm_accuracies)
clm_acc = float(np.mean(clm_accuracies))
clm_bins = compute_reliability_diagram_data(clm_confidences, clm_accuracies)
print(f"\n✅ CLM Done | ECE: {clm_ece:.4f} | Accuracy: {clm_acc:.4f}")

# ── TASK 2: Facts Generation (T-REx) ────────────────────────
print("\n" + "="*60)
print("TASK 2: Facts Generation (T-REx)")
print("="*60)

# Baseline: no instruction prompt
facts_confidences = []
facts_accuracies  = []

facts_start_time = time.time() # Added timer start
for idx, row in trex_eval.iterrows():
    text   = str(row['text']).strip()
    entity = str(row['entity']).strip()

    try:
        entity_tokens = tokenizer(
            entity,
            add_special_tokens=False,
            return_tensors="pt"
        )["input_ids"]

        if entity_tokens.shape[1] == 0:
            continue

        true_first_token = entity_tokens[0, 0].item()

        context_ids = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        )["input_ids"].to(model.device)

        with torch.no_grad():
            out   = model(input_ids=context_ids)
            probs = F.softmax(
                out.logits[:, -1, :].float(), dim=-1
            )

        facts_confidences.append(probs.max(dim=-1).values.item())
        facts_accuracies.append(
            int(probs.argmax(dim=-1).item() == true_first_token)
        )

    except Exception:
        continue

    # Changed 5000 to 1000 for faster updates, added percentage and timer
    if (idx + 1) % 1000 == 0:
        elapsed_mins = (time.time() - facts_start_time) / 60
        pct = (idx + 1) / EVAL_SAMPLES * 100
        print(f"  [{idx+1}/{EVAL_SAMPLES}] ({pct:.1f}%) | Time: {elapsed_mins:.1f}m | "
              f"ECE: {compute_ece(facts_confidences, facts_accuracies):.4f} | "
              f"Acc: {np.mean(facts_accuracies):.4f}")

facts_ece = compute_ece(facts_confidences, facts_accuracies)
facts_acc = float(np.mean(facts_accuracies))
facts_bins = compute_reliability_diagram_data(facts_confidences, facts_accuracies)
print(f"\n✅ Facts Done | ECE: {facts_ece:.4f} | Accuracy: {facts_acc:.4f}")

# ── Save Part 1 Results (intermediate) ───────────────────────
part1_results = {
    "CLM":   {"ECE": clm_ece,   "Accuracy": clm_acc,
               "n_samples": len(clm_confidences),
               "bin_accs": clm_bins[0], "bin_confs": clm_bins[1],
               "bin_sizes": clm_bins[2],
               "confidences": clm_confidences,
               "accuracies":  clm_accuracies},
    "Facts": {"ECE": facts_ece, "Accuracy": facts_acc,
               "n_samples": len(facts_confidences),
               "bin_accs": facts_bins[0], "bin_confs": facts_bins[1],
               "bin_sizes": facts_bins[2],
               "confidences": facts_confidences,
               "accuracies":  facts_accuracies},
}

with open(f'{RESULTS_PATH}/baseline_part1.json', 'w') as f:
    json.dump(part1_results, f, indent=2)

print("\n✅ Part 1 results saved. Now run Part 2 cell.")
print(f"   GPU Memory:")
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f"   GPU {i}: {used:.2f} GB used")

In [ ]:
# ============================================================
# NOTEBOOK 2 PART 2: MMLU Evaluation + Final Save (3k subset)
# Run immediately after Part 1 (same session, no restart)
# ============================================================

import ast
import time

# ── Load MMLU ────────────────────────────────────────────────
print("Loading MMLU datasets...")
mmlu_df     = pd.read_csv(f'{DATA_PATH}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{DATA_PATH}/mmlu_val.csv')

def parse_choices(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except: return x

mmlu_df['choices']     = mmlu_df['choices'].apply(parse_choices)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(parse_choices)

# ── CREATE 3K FIXED SUBSET ───────────────────────────────────
print(f"Original MMLU test: {len(mmlu_df)} samples")
mmlu_df_subset = mmlu_df.sample(n=3000, random_state=SEED).reset_index(drop=True)
print(f"Using fixed 3k subset: {len(mmlu_df_subset)} samples")

print(f"  MMLU val   : {len(mmlu_val_df)} samples")
print(f"  Subjects   : {mmlu_df_subset['subject'].nunique()}")

ANSWER_OPTIONS = ['A', 'B', 'C', 'D']

# Get token IDs for A B C D
answer_token_ids = []
for opt in ANSWER_OPTIONS:
    tid = tokenizer(
        opt, add_special_tokens=False, return_tensors="pt"
    )["input_ids"][0][0].item()
    answer_token_ids.append(tid)
print(f"  Answer token IDs: {dict(zip(ANSWER_OPTIONS, answer_token_ids))}")

def build_mmlu_prompt(row, val_df, n_shots=5):
    subject = row['subject']
    prompt  = (f"The following are multiple choice questions "
               f"(with answers) about {subject}.\\n\\n")

    subject_val = val_df[val_df['subject'] == subject]
    if len(subject_val) < n_shots:
        subject_val = val_df

    shots = subject_val.sample(
        n=min(n_shots, len(subject_val)), random_state=SEED
    )

    for _, shot in shots.iterrows():
        choices = shot['choices']
        prompt += f"Question: {shot['question']}\\n"
        for i, opt in enumerate(ANSWER_OPTIONS):
            prompt += f"{opt}. {choices[i]}\\n"
        prompt += f"Answer: {ANSWER_OPTIONS[int(shot['answer'])]}\\n\\n"

    # Test question (no answer)
    choices = row['choices']
    prompt += f"Question: {row['question']}\\n"
    for i, opt in enumerate(ANSWER_OPTIONS):
        prompt += f"{opt}. {choices[i]}\\n"
    prompt += "Answer:"
    return prompt

# ── TASK 3: MMLU (3K SUBSET) ─────────────────────────────────
print("\\n" + "="*60)
print("TASK 3: MMLU (Language Understanding) - 3K SUBSET")
print("="*60)

mmlu_confidences = []
mmlu_accuracies  = []

mmlu_start_time = time.time()
for idx, row in mmlu_df_subset.iterrows():
    try:
        prompt    = build_mmlu_prompt(row, mmlu_val_df, n_shots=5)
        input_ids = tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=2048
        )["input_ids"].to(model.device)

        with torch.no_grad():
            out    = model(input_ids=input_ids)
            logits = out.logits[:, -1, :]

        answer_logits = torch.tensor(
            [logits[0, tid].item() for tid in answer_token_ids]
        )
        answer_probs = F.softmax(answer_logits.float(), dim=-1)

        mmlu_confidences.append(answer_probs.max().item())
        mmlu_accuracies.append(
            int(answer_probs.argmax().item() == int(row['answer']))
        )

    except Exception:
        continue

    if (idx + 1) % 500 == 0:
        elapsed_mins = (time.time() - mmlu_start_time) / 60
        pct = (idx + 1) / len(mmlu_df_subset) * 100
        print(f"  [{idx+1}/3000] ({pct:.1f}%) | Time: {elapsed_mins:.1f}m | "
              f"ECE: {compute_ece(mmlu_confidences, mmlu_accuracies):.4f} | "
              f"Acc: {np.mean(mmlu_accuracies):.4f}")

mmlu_ece = compute_ece(mmlu_confidences, mmlu_accuracies)
mmlu_acc = float(np.mean(mmlu_accuracies))
mmlu_bins = compute_reliability_diagram_data(mmlu_confidences, mmlu_accuracies)
print(f"\\n✅ MMLU (3K) Done | ECE: {mmlu_ece:.4f} | Accuracy: {mmlu_acc:.4f}")

# ── Merge + Save Final Results ───────────────────────────────
print("\\n" + "="*60)
print("SAVING FINAL BASELINE RESULTS (3K MMLU)")
print("="*60)

with open(f'{RESULTS_PATH}/baseline_part1.json', 'r') as f:
    part1 = json.load(f)

baseline_results = {
    "model"        : "LLaMA-7B (baseline, no fine-tuning, 3K MMLU)",
    "eval_samples" : EVAL_SAMPLES,
    "n_bins"       : N_BINS,
    "CLM"  : part1["CLM"],
    "Facts": part1["Facts"],
    "MMLU" : {
        "ECE"        : mmlu_ece,
        "Accuracy"   : mmlu_acc,
        "n_samples"  : len(mmlu_confidences),
        "subset_size": 3000,
        "subset_seed": SEED,
        "bin_accs"   : mmlu_bins[0],
        "bin_confs"  : mmlu_bins[1],
        "bin_sizes"  : mmlu_bins[2],
        "confidences": mmlu_confidences,
        "accuracies" : mmlu_accuracies,
    },
}

with open(f'{RESULTS_PATH}/baseline_results_3k_mmlu.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

# Clean up intermediate file
os.remove(f'{RESULTS_PATH}/baseline_part1.json')

# ── SAVE SUBSET INDICES FOR FUTURE USE ──────────────────────
subset_indices = mmlu_df_subset.index.tolist()
with open(f'{RESULTS_PATH}/mmlu_3k_subset_indices.json', 'w') as f:
    json.dump({"indices": subset_indices, "seed": SEED}, f, indent=2)
print(f"✅ Saved MMLU 3K subset indices: {RESULTS_PATH}/mmlu_3k_subset_indices.json")

# ── Final Summary ────────────────────────────────────────────
print(f"\\n{'='*60}")
print(f"BASELINE COMPLETE — FINAL SUMMARY (3K MMLU)")
print(f"{'='*60}")
print(f"  Task         | ECE    | Accuracy | Samples")
print(f"  -------------|--------|----------|--------")
print(f"  CLM (PILE)   | "
      f"{baseline_results['CLM']['ECE']:.4f} | "
      f"{baseline_results['CLM']['Accuracy']:.4f}   | "
      f"{baseline_results['CLM']['n_samples']}")
print(f"  Facts (TREx) | "
      f"{baseline_results['Facts']['ECE']:.4f} | "
      f"{baseline_results['Facts']['Accuracy']:.4f}   | "
      f"{baseline_results['Facts']['n_samples']}")
print(f"  MMLU (3K)    | "
      f"{baseline_results['MMLU']['ECE']:.4f} | "
      f"{baseline_results['MMLU']['Accuracy']:.4f}   | "
      f"{baseline_results['MMLU']['n_samples']}")
print(f"{'='*60}")
print(f"\\n✅ Saved to: {RESULTS_PATH}/baseline_results_3k_mmlu.json")
print(f"\\nGPU memory cleared. Ready for Instruction Tuning phase.")

# ── Clear GPU Memory ─────────────────────────────────────────
del model
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f"  GPU {i}: {used:.2f} GB remaining")

In [ ]:
# ============================================================
# VISUALIZE BASELINE LLaMA-7B RESULTS
# Load JSON and create publication-ready plots
# ============================================================

import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Rectangle

# Set style
plt.style.use('default')
sns.set_palette("husl")
figsize = (12, 8)

# ── LOAD RESULTS ─────────────────────────────────────────────
with open('/kaggle/working/results/baseline_results_3k_mmlu.json', 'r') as f:
    results = json.load(f)

# Extract metrics
tasks = ['CLM (PILE)', 'Facts (T-REx)', 'MMLU (3K)']
eces = [results['CLM']['ECE'], results['Facts']['ECE'], results['MMLU']['ECE']]
accs = [results['CLM']['Accuracy'], results['Facts']['Accuracy'], results['MMLU']['Accuracy']]
samples = [results['CLM']['n_samples'], results['Facts']['n_samples'], results['MMLU']['n_samples']]

print("Baseline Metrics:")
for t, e, a, s in zip(tasks, eces, accs, samples):
    print(f"  {t}: ECE={e:.4f}, Acc={a:.4f}, N={s:,}")

# ── PLOT 1: ECE & ACCURACY BAR CHART ────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ECE bars
bars1 = ax1.bar(tasks, eces, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8, edgecolor='black', linewidth=1.2)
ax1.set_ylabel('ECE', fontsize=12, fontweight='bold')
ax1.set_title('Calibration (Lower = Better)', fontsize=14, fontweight='bold', pad=20)
ax1.set_ylim(0, max(eces)*1.3)
for bar, e in zip(bars1, eces):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f'{e:.4f}', 
             ha='center', va='bottom', fontweight='bold')

# Accuracy bars
bars2 = ax2.bar(tasks, accs, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8, edgecolor='black', linewidth=1.2)
ax2.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax2.set_title('Performance (Higher = Better)', fontsize=14, fontweight='bold', pad=20)
ax2.set_ylim(0, max(accs)*1.3)
for bar, a in zip(bars2, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{a:.4f}', 
             ha='center', va='bottom', fontweight='bold')

plt.suptitle('LLaMA-7B Pretrained Baseline Results\n(Phase 1 Complete - Excellent Calibration)', 
             fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('/kaggle/working/results/baseline_summary.png', dpi=300, bbox_inches='tight')
plt.show()

# ── PLOT 2: RELIABILITY DIAGRAM (MMLU) ──────────────────────
mmlu = results['MMLU']
bin_confs = np.array(mmlu['bin_confs'])
bin_accs = np.array(mmlu['bin_accs'])
bin_sizes = np.array(mmlu['bin_sizes'])

fig, ax = plt.subplots(figsize=(8, 6))
perfect = np.linspace(0, 1, len(bin_confs))

# Plot reliability curve
ax.plot(bin_confs, bin_accs, 'o-', linewidth=3, markersize=8, color='#d62728', label='LLaMA-7B Baseline')
ax.plot(perfect, perfect, 'k--', linewidth=2, alpha=0.7, label='Perfect Calibration')
ax.fill_between(perfect, perfect, alpha=0.1, color='gray')

# Error bars
ax.errorbar(bin_confs, bin_accs, yerr=np.sqrt(bin_accs*(1-bin_accs)/bin_sizes), 
            fmt='none', ecolor='gray', alpha=0.5, capsize=3)

ax.set_xlabel('Confidence', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title(f'MMLU Reliability Diagram\nECE={mmlu["ECE"]:.4f}, Acc={mmlu["Accuracy"]:.4f} (N={mmlu["n_samples"]})', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('/kaggle/working/results/mmlu_reliability_diagram.png', dpi=300, bbox_inches='tight')
plt.show()

# ── SUMMARY TABLE ───────────────────────────────────────────
print("\n" + "="*60)
print("BASELINE VISUALIZATION COMPLETE")
print("="*60)
print("✅ baseline_summary.png - ECE/Acc bars")
print("✅ mmlu_reliability_diagram.png - Reliability curve")
print("📁 Saved in: /kaggle/working/results/")
print("\n🎉 Phase 1 validated! Ready for Instruction Tuning (Alpaca).")
print("="*60)

# instruction tuning alpaca

In [ ]:
!pip install --upgrade trl

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

In [ ]:
import os, gc, json, ast, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

SEED         = 42
EVAL_SAMPLES = 30000
N_BINS       = 10
EPOCHS       = 3

MODEL_PATH   = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1'
DATA_PATH    = '/kaggle/working/datasets'
RESULTS_PATH = '/kaggle/working/results'
CKPT_PATH    = '/kaggle/working/checkpoints'

os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(CKPT_PATH,    exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
print("All imports done.")

In [ ]:
def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

def parse_choices(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except: return x

print("ECE utilities ready.")

In [ ]:
print("Loading evaluation datasets...")

pile_df     = pd.read_csv(f'{DATA_PATH}/pile_50k.csv')
trex_df     = pd.read_csv(f'{DATA_PATH}/trex_50k.csv')
mmlu_df     = pd.read_csv(f'{DATA_PATH}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{DATA_PATH}/mmlu_val.csv')

mmlu_df['choices']     = mmlu_df['choices'].apply(parse_choices)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(parse_choices)

pile_eval = pile_df.sample(n=EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)
trex_eval = trex_df.sample(n=EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

ANSWER_OPTIONS   = ['A', 'B', 'C', 'D']
PILE_PROMPT      = "Finish this sentence: "
TREX_PROMPT      = "Finish this Wikipedia description: "

print(f"  PILE eval  : {len(pile_eval)}")
print(f"  T-REx eval : {len(trex_eval)}")
print(f"  MMLU test  : {len(mmlu_df)}")
print(f"  MMLU val   : {len(mmlu_val_df)}")
print("Datasets ready.")

In [ ]:
def evaluate_model(model, tokenizer, label="", include_prompt=True):
    """
    Runs all 3 tasks. include_prompt=True for alignment stage (paper Table 1).
    Returns dict with ECE + Accuracy for each task.
    """
    model.eval()
    results = {}

    pile_prompt  = PILE_PROMPT  if include_prompt else ""
    trex_prompt  = TREX_PROMPT  if include_prompt else ""

    # ── TASK 1: CLM ──────────────────────────────────────────
    print(f"\n  [{label}] CLM (PILE)...")
    confs, accs = [], []

    for idx, row in pile_eval.iterrows():
        text = pile_prompt + str(row['text']).strip()
        try:
            input_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"]
            seq_len = input_ids.shape[1]
            if seq_len < 3: continue

            pos        = np.random.randint(1, seq_len)
            context    = input_ids[:, :pos].to(model.device)
            true_token = input_ids[:, pos].item()

            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context).logits[:, -1, :].float(),
                    dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_token))
        except: continue

        if (idx+1) % 10000 == 0:
            print(f"    {idx+1}/{EVAL_SAMPLES} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['CLM'] = {
        'ECE': compute_ece(confs, accs),
        'Accuracy': float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ CLM ECE: {results['CLM']['ECE']:.4f} | "
          f"Acc: {results['CLM']['Accuracy']:.4f}")

    # ── TASK 2: Facts ─────────────────────────────────────────
    print(f"\n  [{label}] Facts Generation (T-REx)...")
    confs, accs = [], []

    for idx, row in trex_eval.iterrows():
        text   = trex_prompt + str(row['text']).strip()
        entity = str(row['entity']).strip()
        try:
            entity_toks = tokenizer(
                entity, add_special_tokens=False, return_tensors="pt"
            )["input_ids"]
            if entity_toks.shape[1] == 0: continue
            true_first = entity_toks[0, 0].item()

            context_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"].to(model.device)

            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context_ids).logits[:, -1, :].float(),
                    dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_first))
        except: continue

        if (idx+1) % 10000 == 0:
            print(f"    {idx+1}/{EVAL_SAMPLES} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['Facts'] = {
        'ECE': compute_ece(confs, accs),
        'Accuracy': float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ Facts ECE: {results['Facts']['ECE']:.4f} | "
          f"Acc: {results['Facts']['Accuracy']:.4f}")

    # ── TASK 3: MMLU ──────────────────────────────────────────
    print(f"\n  [{label}] MMLU...")
    confs, accs = [], []

    answer_token_ids = [
        tokenizer(opt, add_special_tokens=False,
                  return_tensors="pt")["input_ids"][0][0].item()
        for opt in ANSWER_OPTIONS
    ]

    def build_prompt(row):
        subject     = row['subject']
        prompt      = (f"The following are multiple choice questions "
                       f"(with answers) about {subject}.\n\n")
        subject_val = mmlu_val_df[mmlu_val_df['subject'] == subject]
        if len(subject_val) < 5:
            subject_val = mmlu_val_df
        shots = subject_val.sample(n=min(5, len(subject_val)),
                                   random_state=SEED)
        for _, shot in shots.iterrows():
            ch = shot['choices']
            prompt += f"Question: {shot['question']}\n"
            for i, opt in enumerate(ANSWER_OPTIONS):
                prompt += f"{opt}. {ch[i]}\n"
            prompt += f"Answer: {ANSWER_OPTIONS[int(shot['answer'])]}\n\n"
        ch = row['choices']
        prompt += f"Question: {row['question']}\n"
        for i, opt in enumerate(ANSWER_OPTIONS):
            prompt += f"{opt}. {ch[i]}\n"
        prompt += "Answer:"
        return prompt

    for idx, row in mmlu_df.iterrows():
        try:
            input_ids = tokenizer(
                build_prompt(row), return_tensors="pt",
                truncation=True, max_length=2048
            )["input_ids"].to(model.device)

            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, -1, :]

            ans_probs = F.softmax(
                torch.tensor([logits[0, tid].item()
                              for tid in answer_token_ids]).float(),
                dim=-1
            )
            confs.append(ans_probs.max().item())
            accs.append(int(ans_probs.argmax().item() == int(row['answer'])))
        except: continue

        if (idx+1) % 2000 == 0:
            print(f"    {idx+1}/{len(mmlu_df)} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['MMLU'] = {
        'ECE': compute_ece(confs, accs),
        'Accuracy': float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ MMLU ECE: {results['MMLU']['ECE']:.4f} | "
          f"Acc: {results['MMLU']['Accuracy']:.4f}")

    return results

print("Evaluation function ready.")

In [ ]:
print("Loading Alpaca dataset...")
alpaca_df = pd.read_csv(f'{DATA_PATH}/alpaca/alpaca_sampled.csv')
alpaca_dataset = Dataset.from_pandas(alpaca_df[['text']])
print(f"Alpaca samples: {len(alpaca_dataset)}")
print(f"Sample:\n{alpaca_df['text'].iloc[0][:200]}")

In [ ]:
print("Loading LLaMA-7B...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Apply LoRA — matches paper Table 2 exactly
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model + LoRA ready.")

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=f'{CKPT_PATH}/alpaca_lora',
    num_train_epochs=1,                 # <--- CHANGED TO 1 EPOCH
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch = 4*8 = 32 per GPU
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",              # save checkpoint after each epoch
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    
    # --- FIXED NAMING ---
    dataset_text_field="text",          
    max_length=2048,                
)

trainer = SFTTrainer(
    model=model,
    train_dataset=alpaca_dataset,
    args=training_args,
)

print("Starting Alpaca LoRA training...")
trainer.train()
print("Training complete.")

In [ ]:
# Save the trained model immediately
save_path = f'{CKPT_PATH}/alpaca_lora_epoch1'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model saved to: {save_path}")

# Verify files are there
import os
files = os.listdir(save_path)
print(f"Saved files: {files}")

In [ ]:
# Continue training for 2 more epochs from epoch 1 checkpoint
training_args = SFTConfig(
    output_dir=f'{CKPT_PATH}/alpaca_lora',
    num_train_epochs=2,                 # 2 more epochs
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    dataset_text_field="text",
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,                        # already in memory, has epoch 1 weights
    train_dataset=alpaca_dataset,
    args=training_args,
)

print("Continuing training — Epochs 2 and 3...")
trainer.train()
print("Done.")

In [ ]:
import os
ckpt_base = f'{CKPT_PATH}/alpaca_lora'
print("Available checkpoints:")
for item in sorted(os.listdir(ckpt_base)):
    print(f"  {item}")
print(f"\nManually saved:")
print(f"  epoch1: {os.path.exists(f'{CKPT_PATH}/alpaca_lora_epoch1')}")
print(f"  epoch3: {os.path.exists(f'{CKPT_PATH}/alpaca_lora_epoch3')}")

# Instruction tuning OA

In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

In [3]:
import os, gc, json, ast, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

SEED         = 42
EVAL_CLM     = 5000
EVAL_FACTS   = 5000
EVAL_MMLU    = 3000
N_BINS       = 10
EPOCHS       = 3

MODEL_PATH   = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1'
DATA_PATH    = '/kaggle/working/datasets'
RESULTS_PATH = '/kaggle/working/results'

os.makedirs(RESULTS_PATH, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
print("✅ Done.")

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB
✅ Done.


In [4]:
def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0: continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

def parse_choices(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except: return x

print("✅ ECE utilities ready.")

✅ ECE utilities ready.


In [5]:
print("Loading datasets...")

pile_df     = pd.read_csv(f'{DATA_PATH}/pile_50k.csv')
trex_df     = pd.read_csv(f'{DATA_PATH}/trex_50k.csv')
mmlu_df     = pd.read_csv(f'{DATA_PATH}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{DATA_PATH}/mmlu_val.csv')
oa_df       = pd.read_csv(f'{DATA_PATH}/oa/oa_data.csv')

mmlu_df['choices']     = mmlu_df['choices'].apply(parse_choices)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(parse_choices)

pile_eval = pile_df.sample(n=EVAL_CLM,   random_state=SEED).reset_index(drop=True)
trex_eval = trex_df.sample(n=EVAL_FACTS, random_state=SEED).reset_index(drop=True)
mmlu_eval = mmlu_df.sample(n=EVAL_MMLU,  random_state=SEED).reset_index(drop=True)
oa_dataset = Dataset.from_pandas(oa_df[['text']])

ANSWER_OPTIONS = ['A', 'B', 'C', 'D']
PILE_PROMPT    = "Finish this sentence: "
TREX_PROMPT    = "Finish this Wikipedia description: "

print(f"✅ PILE eval  : {len(pile_eval)}")
print(f"✅ T-REx eval : {len(trex_eval)}")
print(f"✅ MMLU eval  : {len(mmlu_eval)}")
print(f"✅ OA train   : {len(oa_dataset)}")

Loading datasets...
✅ PILE eval  : 5000
✅ T-REx eval : 5000
✅ MMLU eval  : 3000
✅ OA train   : 11000


In [6]:
def evaluate_model(model, tokenizer, label=""):
    model.eval()
    results     = {}
    total_start = time.time()

    # ── CLM ──────────────────────────────────────────────────
    print(f"\n  [{label}] CLM (PILE {EVAL_CLM} samples)...")
    task_start  = time.time()
    confs, accs = [], []

    for idx, row in pile_eval.iterrows():
        text = PILE_PROMPT + str(row['text']).strip()
        try:
            input_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"]
            seq_len = input_ids.shape[1]
            if seq_len < 3: continue
            pos        = np.random.randint(1, seq_len)
            context    = input_ids[:, :pos].to(model.device)
            true_token = input_ids[:, pos].item()
            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context).logits[:, -1, :].float(),
                    dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_token))
        except: continue

        if (idx+1) % 5000 == 0:
            elapsed   = time.time() - task_start
            rate      = (idx+1) / elapsed
            remaining = (EVAL_CLM - (idx+1)) / rate
            print(f"    {idx+1}/{EVAL_CLM} | "
                  f"ECE: {compute_ece(confs,accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f} | "
                  f"Elapsed: {elapsed/60:.1f}m | "
                  f"ETA: {remaining/60:.1f}m")

    results['CLM'] = {
        'ECE':       compute_ece(confs, accs),
        'Accuracy':  float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ CLM ECE: {results['CLM']['ECE']:.4f} | "
          f"Acc: {results['CLM']['Accuracy']:.4f} | "
          f"Time: {(time.time()-task_start)/60:.1f}m")

    # ── Facts ─────────────────────────────────────────────────
    print(f"\n  [{label}] Facts (T-REx {EVAL_FACTS} samples)...")
    task_start  = time.time()
    confs, accs = [], []

    for idx, row in trex_eval.iterrows():
        text   = TREX_PROMPT + str(row['text']).strip()
        entity = str(row['entity']).strip()
        try:
            entity_toks = tokenizer(
                entity, add_special_tokens=False,
                return_tensors="pt"
            )["input_ids"]
            if entity_toks.shape[1] == 0: continue
            true_first  = entity_toks[0, 0].item()
            context_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"].to(model.device)
            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context_ids).logits[:, -1, :].float(),
                    dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_first))
        except: continue

        if (idx+1) % 5000 == 0:
            elapsed   = time.time() - task_start
            rate      = (idx+1) / elapsed
            remaining = (EVAL_FACTS - (idx+1)) / rate
            print(f"    {idx+1}/{EVAL_FACTS} | "
                  f"ECE: {compute_ece(confs,accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f} | "
                  f"Elapsed: {elapsed/60:.1f}m | "
                  f"ETA: {remaining/60:.1f}m")

    results['Facts'] = {
        'ECE':       compute_ece(confs, accs),
        'Accuracy':  float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ Facts ECE: {results['Facts']['ECE']:.4f} | "
          f"Acc: {results['Facts']['Accuracy']:.4f} | "
          f"Time: {(time.time()-task_start)/60:.1f}m")

    # ── MMLU ──────────────────────────────────────────────────
    print(f"\n  [{label}] MMLU ({EVAL_MMLU} samples)...")
    task_start  = time.time()
    confs, accs = [], []

    answer_token_ids = [
        tokenizer(opt, add_special_tokens=False,
                  return_tensors="pt")["input_ids"][0][0].item()
        for opt in ANSWER_OPTIONS
    ]

    def build_prompt(row):
        subject     = row['subject']
        prompt      = (f"The following are multiple choice questions "
                       f"(with answers) about {subject}.\n\n")
        subject_val = mmlu_val_df[mmlu_val_df['subject'] == subject]
        if len(subject_val) < 5: subject_val = mmlu_val_df
        shots = subject_val.sample(
            n=min(5, len(subject_val)), random_state=SEED
        )
        for _, shot in shots.iterrows():
            ch = shot['choices']
            prompt += f"Question: {shot['question']}\n"
            for i, opt in enumerate(ANSWER_OPTIONS):
                prompt += f"{opt}. {ch[i]}\n"
            prompt += f"Answer: {ANSWER_OPTIONS[int(shot['answer'])]}\n\n"
        ch = row['choices']
        prompt += f"Question: {row['question']}\n"
        for i, opt in enumerate(ANSWER_OPTIONS):
            prompt += f"{opt}. {ch[i]}\n"
        prompt += "Answer:"
        return prompt

    for idx, row in mmlu_eval.iterrows():
        try:
            input_ids = tokenizer(
                build_prompt(row), return_tensors="pt",
                truncation=True, max_length=2048
            )["input_ids"].to(model.device)
            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, -1, :]
            ans_probs = F.softmax(
                torch.tensor([logits[0, tid].item()
                              for tid in answer_token_ids]).float(),
                dim=-1
            )
            confs.append(ans_probs.max().item())
            accs.append(int(ans_probs.argmax().item() == int(row['answer'])))
        except: continue

        if (idx+1) % 500 == 0:
            elapsed   = time.time() - task_start
            rate      = (idx+1) / elapsed
            remaining = (EVAL_MMLU - (idx+1)) / rate
            print(f"    {idx+1}/{EVAL_MMLU} | "
                  f"ECE: {compute_ece(confs,accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f} | "
                  f"Elapsed: {elapsed/60:.1f}m | "
                  f"ETA: {remaining/60:.1f}m")

    results['MMLU'] = {
        'ECE':       compute_ece(confs, accs),
        'Accuracy':  float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ MMLU ECE: {results['MMLU']['ECE']:.4f} | "
          f"Acc: {results['MMLU']['Accuracy']:.4f} | "
          f"Time: {(time.time()-task_start)/60:.1f}m")

    print(f"\n  ⏱ Total eval time: {(time.time()-total_start)/60:.1f}m")
    return results

print("✅ Evaluation function ready.")

✅ Evaluation function ready.


In [ ]:
print("Loading LLaMA-7B...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Model + LoRA ready.")

for restart run 

In [7]:
print("Loading LLaMA-7B...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

# This loads the pure, untouched base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("✅ PURE Base Model ready! (We intentionally skipped the blank LoRA config).")

Loading LLaMA-7B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ PURE Base Model ready! (We intentionally skipped the blank LoRA config).


In [8]:
from peft import PeftModel

OA_DIR = '/kaggle/working/oa_lora_checkpoints'
LATEST_CHECKPOINT = f"{OA_DIR}/checkpoint-1032"

print(f"Loading trained LoRA adapters from: {LATEST_CHECKPOINT}...")

# This snaps your TRAINED weights onto the pure base model
model = PeftModel.from_pretrained(model, LATEST_CHECKPOINT)

print("✅ Custom adapters successfully attached! You are ready for Phase 3.")

Loading trained LoRA adapters from: /kaggle/working/oa_lora_checkpoints/checkpoint-1032...
✅ Custom adapters successfully attached! You are ready for Phase 3.


for restart session use the above both code


In [ ]:
import os

# THIS IS THE CRUCIAL LINE YOU NEED:
OA_DIR = '/kaggle/working/oa_lora_checkpoints'

ckpts = sorted([d for d in os.listdir(OA_DIR) if d.startswith('checkpoint')])
print(f"Found checkpoints: {ckpts}")

In [ ]:
from peft import PeftModel

# Point directly to your final epoch's folder
LATEST_CHECKPOINT = f"{OA_DIR}/checkpoint-1032"

print(f"Loading LoRA adapters from: {LATEST_CHECKPOINT}...")

# This command snaps your trained weights onto the base LLaMA model
model = PeftModel.from_pretrained(model, LATEST_CHECKPOINT)

print("✅ Custom adapters successfully attached to the base model! You are ready for Phase 3.")

In [ ]:
import time
from trl import SFTTrainer, SFTConfig

# We will use this exact same folder for all our steps
OA_DIR = '/kaggle/working/oa_lora_checkpoints'

print("="*60)
print("PHASE 1: TRAINING EPOCH 1")
print("="*60)
train_start = time.time()

training_args_ep1 = SFTConfig(
    output_dir=OA_DIR,
    num_train_epochs=1,                 # Only train 1 epoch for now
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",              # CRITICAL: Saves a checkpoint at the end of the epoch
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    dataset_text_field="text",
    max_length=2048,
)

trainer_ep1 = SFTTrainer(
    model=model,
    train_dataset=oa_dataset,
    args=training_args_ep1,
)

trainer_ep1.train()

print(f"✅ Epoch 1 Complete and Saved! | Time: {(time.time()-train_start)/60:.1f}m")

In [ ]:
print("="*60)
print("PHASE 2: RESUMING FOR EPOCHS 2 AND 3")
print("="*60)
train_start = time.time()

training_args_ep3 = SFTConfig(
    output_dir=OA_DIR,
    num_train_epochs=3,                 # Total epochs we want to reach (it knows 1 is done!)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",              # Saves checkpoints for ep 2 and ep 3
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    dataset_text_field="text",
    max_length=2048,
)

trainer_ep3 = SFTTrainer(
    model=model,
    train_dataset=oa_dataset,
    args=training_args_ep3,
)

# CRITICAL: resume_from_checkpoint=True tells it to load Epoch 1's saved data
trainer_ep3.train(resume_from_checkpoint=True) 

print(f"✅ Epochs 2 and 3 Complete and Saved! | Time: {(time.time()-train_start)/60:.1f}m")

In [ ]:
import json

print("="*60)
print("PHASE 3: FINAL EVALUATION")
print("="*60)

# Run your custom evaluation function on the fully trained model
final_results = evaluate_model(
    model, 
    tokenizer,
    label="OA_LoRA_Final_Epoch3"
)

# Format it cleanly into a dictionary
all_results = {
    "model": "LLaMA-7B", 
    "method": "OA_LoRA", 
    "final_evaluation": final_results
}

# Save the JSON file
with open(f'{RESULTS_PATH}/oa_lora_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("\n" + "="*60)
print("OA LORA — FINAL SUMMARY (EPOCH 3)")
print("="*60)
print(f"{'Task':<8} {'ECE':>8} {'Accuracy':>10}")
print("-" * 30)

for task in ['CLM', 'Facts', 'MMLU']:
    print(f"{task:<8} "
          f"{final_results[task]['ECE']:>8.4f} "
          f"{final_results[task]['Accuracy']:>10.4f}")
          
print("="*60)
print(f"✅ Final results saved to: {RESULTS_PATH}/oa_lora_results.json")

In [9]:
import json
import time

print("\n" + "="*60)
print("PHASE 3: FINAL EVALUATION (EPOCH 3)")
print("="*60)

# Start global stopwatch
total_start = time.time()

# 1. Run the evaluation
# Note: Your custom 'evaluate_model' function already has internal logic 
# to print progress every 5000/500 samples, so you will see ETA updates 
# in your console as it runs!
final_results = evaluate_model(model, tokenizer, label="OA_LoRA_Epoch3")

# 2. Format and Save Results
all_results = {
    "model": "LLaMA-7B", 
    "method": "OA_LoRA", 
    "epochs": {
        "epoch_3": final_results
    }
}

with open(f'{RESULTS_PATH}/oa_lora_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# 3. Final Time Calculation
total_elapsed_mins = (time.time() - total_start) / 60

# 4. Final Scorecard
print("\n" + "="*60)
print("OA LORA — FINAL SUMMARY (EPOCH 3)")
print(f"Total Time Taken: {total_elapsed_mins:.1f} minutes")
print("="*60)
print(f"{'Task':<8} {'ECE':>8} {'Accuracy':>10}")
print("-" * 35)

for task in ['CLM', 'Facts', 'MMLU']:
    print(f"{task:<8} "
          f"{final_results[task]['ECE']:>8.4f} "
          f"{final_results[task]['Accuracy']:>10.4f}")
          
print("="*60)
print(f"✅ Final results successfully saved to: {RESULTS_PATH}/oa_lora_results.json")


PHASE 3: FINAL EVALUATION (EPOCH 3)

  [OA_LoRA_Epoch3] CLM (PILE 5000 samples)...
    5000/5000 | ECE: 0.0262 | Acc: 0.5880 | Elapsed: 17.0m | ETA: 0.0m
    ✅ CLM ECE: 0.0262 | Acc: 0.5880 | Time: 17.0m

  [OA_LoRA_Epoch3] Facts (T-REx 5000 samples)...
    5000/5000 | ECE: 0.0813 | Acc: 0.0974 | Elapsed: 6.5m | ETA: 0.0m
    ✅ Facts ECE: 0.0813 | Acc: 0.0974 | Time: 6.5m

  [OA_LoRA_Epoch3] MMLU (3000 samples)...
    500/3000 | ECE: 0.0813 | Acc: 0.4200 | Elapsed: 5.9m | ETA: 29.6m
    1000/3000 | ECE: 0.0707 | Acc: 0.4260 | Elapsed: 11.9m | ETA: 23.7m
    1500/3000 | ECE: 0.0795 | Acc: 0.4200 | Elapsed: 17.6m | ETA: 17.6m
    2000/3000 | ECE: 0.0773 | Acc: 0.4240 | Elapsed: 23.4m | ETA: 11.7m
    2500/3000 | ECE: 0.0727 | Acc: 0.4260 | Elapsed: 29.4m | ETA: 5.9m
    3000/3000 | ECE: 0.0705 | Acc: 0.4310 | Elapsed: 35.4m | ETA: 0.0m
    ✅ MMLU ECE: 0.0705 | Acc: 0.4310 | Time: 35.4m

  ⏱ Total eval time: 58.9m

OA LORA — FINAL SUMMARY (EPOCH 3)
Total Time Taken: 58.9 minutes
Task    